 # PolicyProof AI — Stage 3: AlignTune QLoRA T4

Research-only integration test. Run in a **new Google Colab T4 notebook**. Use your downloaded CuratorKIT ZIP, which contains only fictional training rows. The previous 48-case benchmark must never enter training. **Installation and GPU training default to OFF.** This is not a verified model improvement. Tiny Aya's CC-BY-NC license applies.

## 1. Check T4 and pull versioned project code.

In [ ]:
import torch, subprocess, pathlib, os, sys, json, time
assert torch.cuda.is_available(), "Select a T4 GPU under Runtime settings and connect."
print("GPU:", torch.cuda.get_device_name(0))
ROOT=pathlib.Path("/content/PolicyProof-AI")
if not ROOT.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/betterlifestyle04-commits/PolicyProof-AI.git",str(ROOT)],check=True)
else:
    subprocess.run(["git","-C",str(ROOT),"pull","--ff-only"],check=True)
os.chdir(ROOT)
sys.path.insert(0,str(ROOT))
REV=subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip()
print("Code revision:",REV)


## 2. Upload the CuratorKIT training ZIP and verify its integrity before extraction. Only allowlisted files are extracted.

In [ ]:
from google.colab import files
from zipfile import ZipFile
from scripts.verify_curator_archive import check_zip
uploaded=files.upload()
assert len(uploaded)==1, "Upload only the policyproof_curatorkit_train_v1.zip"
zip_name=next(iter(uploaded))
assert zip_name.endswith(".zip")
report=check_zip(zip_name)
assert report["status"]=="pass" and report["row_count"]==78
print(json.dumps(report,indent=2,ensure_ascii=False))
CURATED=ROOT/"artifacts"/"verified_curator_train"
CURATED.mkdir(parents=True,exist_ok=True)
allowed={"sft_alpaca.jsonl","sft_sharegpt.jsonl","manifest.json",
         "lexsi_provenance.json","checksums.txt","README.md","dataset_card.md","rejected.jsonl"}
with ZipFile(zip_name) as z:
    for name in sorted(allowed.intersection(z.namelist())):
        (CURATED/name).write_bytes(z.read(name))


## 3. Authenticate through the Hugging Face login UI and record the actual model SHA. Do not paste tokens into notebook cells.

In [ ]:
## 6. Optional experimental 10-step 4-bit LoRA run — OFF by default

Only enable after Step 5 prints `CONFIG_VALIDATED: True`. Model weights will be loaded and GPU memory consumed. Completion would prove a small training run works, not that quality improves. Save adapter before the Colab instance terminates.

## 4. Verify AlignTune and the actual training dependencies

The `pip check` output can include unrelated preinstalled Colab packages (Google ADK, RAPIDS, IPython). Do not downgrade those blindly. This cell prints all `pip check` details, then runs a **separate-process smoke test** of AlignTune's trainer factory, Transformers, PEFT, TRL, BitsAndBytes and CUDA.

A successful import test is only a prerequisite; it is **not** proof Tiny Aya's trainer configuration will work or fit in VRAM. Keep `RUN_TRAINING=False` until the test is reviewed.

In [ ]:
import subprocess, sys, json
from importlib.metadata import version, PackageNotFoundError

INSTALL_NOW = False
if INSTALL_NOW:
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "aligntune>=0.1.10,<1", "huggingface-hub>=1.31,<2"],
        check=True
    )

check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True, text=True
)
print("GLOBAL PIP CHECK STATUS:", check.returncode)
print(check.stdout or check.stderr or "(no output)")

for package in ["aligntune", "transformers", "datasets", "huggingface-hub",
                "trl", "peft", "torch", "bitsandbytes"]:
    try:
        print(package + ":", version(package))
    except PackageNotFoundError:
        print(package + ": NOT INSTALLED")

test_source = '''
import json
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer
import bitsandbytes as bnb
from datasets import load_dataset
from aligntune.core.backend_factory import create_sft_trainer
assert callable(create_sft_trainer)
assert torch.cuda.is_available(), "No CUDA GPU"
v=torch.empty((1,), device="cuda")
torch.cuda.synchronize()
print(json.dumps({"imports":"ok", "cuda":"ok", "gpu":torch.cuda.get_device_name(0),
                  "bnb_4bit":hasattr(bnb,"nn"), "trainer_factory":"ok"}))
'''
print("\nTRAINING LIBRARY IMPORT SMOKE TEST:")
smoke = subprocess.run(
    [sys.executable, "-c", test_source],
    capture_output=True, text=True, timeout=180
)
print("Exit code:", smoke.returncode)
print("stdout:\n", smoke.stdout[-4500:])
print("stderr (last 6000 chars):\n", smoke.stderr[-6000:])
TRAINING_ENV_READY = smoke.returncode == 0 and '"imports": "ok"' in smoke.stdout
print("TRAINING LIBRARIES READY:", TRAINING_ENV_READY)
if not TRAINING_ENV_READY:
    print("Do not run training. Share the smoke-test traceback for targeted repair.")
else:
    print("Core libraries import; inspect model-specific QLoRA compatibility before training.")


## 5. Dry-run the exact AlignTune training configuration (no model weights)

After validating CuratorKIT ZIP extraction and successfully importing AlignTune, run this cell to validate the 78 training records, ensure no held-out policy IDs are present, instantiate AlignTune's **TRL trainer wrapper only**, and check the LoRA + 4-bit settings. This does **not** download a model or train.

In [ ]:
import subprocess, sys, json
from pathlib import Path
train_path = CURATED / "sft_alpaca.jsonl"
assert train_path.exists(), "Run archive upload/validation first"
config_only=subprocess.run(
    [sys.executable, "-m", "scripts.check_align_config",
     "--curated-dir", str(CURATED)],
    capture_output=True, text=True, timeout=180,
)
print("AlignTune CONFIG-ONLY CHECK EXIT:", config_only.returncode)
print(config_only.stdout[-8500:])
print(config_only.stderr[-5000:])
CONFIG_VALIDATED = config_only.returncode == 0 and "config_only_valid" in config_only.stdout
print("CONFIG_VALIDATED:", CONFIG_VALIDATED)
if not CONFIG_VALIDATED:
    print("STOP. Share the error output; do not start model training.")


## 6. Experimental LoRA training — disabled until all dependency conflicts are resolved

`TRAINING_ENV_READY` must be `True` and the AlignTune trainer factory must import. Set `RUN_TRAINING=True` only after both conditions are verified. The LoRA configuration for Tiny Aya is experimental and may still require model-specific adaptation.

In [ ]:
RUN_TRAINING=False
if RUN_TRAINING:
    assert CONFIG_VALIDATED, "Training configuration-only preflight failed. Do not train yet."
    from datasets import load_dataset
    from aligntune.core.backend_factory import create_sft_trainer
    loaded=load_dataset("json",data_files={"train":str(train_path)},split="train")
    assert len(loaded)==78
    OUTPUT=ROOT/"artifacts"/"align_qlora_smoke"
    trainer=create_sft_trainer(
        model_name=MODEL_ID,
        dataset_name=str(CURATED),
        config_name="sft_alpaca",
        backend="trl",
        task_type="instruction_following",
        split="train",
        output_dir=str(OUTPUT),
        num_epochs=1,
        max_steps=10,
        batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        max_seq_length=1024,
        instruction_column="instruction",
        input_column="input",
        response_column="output",
        peft_enabled=True,
        lora_r=8,lora_alpha=16,lora_dropout=0.05,
        lora_target_modules=["q_proj","k_proj","v_proj","o_proj"],
        quantization={"load_in_4bit":True,
                      "bnb_4bit_compute_dtype":"float16",
                      "bnb_4bit_quant_type":"nf4"},
        use_gradient_checkpointing=True,
        use_flash_attention_2=False,
        attn_implementation="sdpa",
    )
    t0=time.time()
    trainer.train()
    adapter=trainer.save_model(output_dir=str(OUTPUT/"adapter"))
    RUN_INFO={"trained":True,"model":MODEL_ID,"base_sha":MODEL_SHA,
        "code_sha":REV,"train_data_sha":sha256(train_path),
        "train_rows":78,"duration_s":round(time.time()-t0,1),
        "adapter_location":str(adapter),"steps":10}
    (OUTPUT/"run_info.json").write_text(json.dumps(RUN_INFO,indent=2),encoding="utf-8")
    print("Trained and saved small integration adapter:",adapter)
else:
    print("Training OFF. No GPU training was executed.")


## 7. Export only after a successful training run. Do not label a failed or unexecuted run as successful.

In [ ]:
if RUN_TRAINING and "RUN_INFO" in globals() and RUN_INFO.get("trained"):
    import shutil
    archive=shutil.make_archive("/content/policyproof_align_smoke","zip",root_dir=OUTPUT)
    files.download(archive)
else:
    print("No trained checkpoint to download")


## Next stage

If the install fails, share the traceback without credentials. If it works, download the adapter plus run_info evidence. Then build a fresh uninspected multilingual benchmark and verify the tuned model independently. Official docs: https://aligntune.lexsi.ai/user-guide/sft/ .